# Transformer
"Attention Is All You Need" (Vaswani et al., 2017)

**Idea:** a sequence-to-sequence model with **no recurrence and no convolution**. An **encoder** reads the whole source sentence with self-attention; a **decoder** writes the target one token at a time, attending to what it has written so far (masked self-attention) and to the encoder's output (**cross-attention**).

### Transformer vs what came before and after
| | RNN seq2seq + attention (`lstm.ipynb`, `gru.ipynb`) | **Transformer (2017)** | BERT (`bert.ipynb`) | GPT / SLM (`slm.ipynb`) |
|---|---|---|---|---|
| Parts | RNN encoder + RNN decoder | **encoder + decoder** | encoder only | decoder only |
| Sequence mixing | recurrence, step by step | self-attention, all pairs at once | self-attention | causal self-attention |
| Path between two tokens | $O(T)$ steps | $O(1)$ | $O(1)$ | $O(1)$ |
| Training over time | sequential | parallel | parallel | parallel |
| Source -> target link | attention over encoder states | **cross-attention** in every decoder layer | - | none (one sequence) |
| Position | implicit | **sin/cos added** to the input | learned | learned / RoPE (`rope.ipynb`) |
| Norm placement | - | **Post-LN** | Post-LN | Pre-LN |
| Task | translation | translation | understanding | generation |

Here: the full architecture as in the paper (Post-LN, sinusoidal positions, ReLU FFN, shared embeddings, label smoothing, the "Noam" learning-rate schedule), small, trained on a synthetic translation task: human-written dates -> ISO format.

Shape flow: `src (B, S) -> embed + PE (B, S, D) -> [self-attn -> FFN] x N -> memory (B, S, D)` and `tgt (B, T) -> embed + PE -> [masked self-attn -> cross-attn(memory) -> FFN] x N -> logits (B, T, V)`

In [ ]:
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
random.seed(42)

## 1. Data - date "translation"
No translation corpus is needed to exercise every part of the model. Source: a date written one of several human ways; target: the same date as `YYYY-MM-DD`.

`tuesday, march 5, 2019 -> 2019-03-05` &nbsp;&nbsp; `05/03/19 -> 2019-03-05` &nbsp;&nbsp; `5 mar 2019 -> 2019-03-05`

The model has to find the right source characters for each output position (alignment, a job for cross-attention), map month names to numbers, and reorder. Character-level, one shared vocabulary for source and target (the paper also shares one BPE vocabulary between both languages).

Special tokens: `<pad>` (0), `<bos>` (1, starts the target), `<eos>` (2, ends it).

In [ ]:
MONTHS = ["january", "february", "march", "april", "may", "june", "july", "august", "september", "october", "november", "december"]
DAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

def make_pair():
    y, m, d = random.randint(1950, 2049), random.randint(1, 12), random.randint(1, 28)
    month, mon, wd = MONTHS[m - 1], MONTHS[m - 1][:3], random.choice(DAYS)     # weekday is a distractor
    src = random.choice([
        f"{d} {month} {y}", f"{month} {d}, {y}", f"{d} {mon} {y}", f"{mon} {d} {y}",
        f"{wd}, {month} {d}, {y}", f"{d:02d}/{m:02d}/{y % 100:02d}", f"{d}.{m}.{y}", f"{month} {d} {y}",
    ])
    return src, f"{y:04d}-{m:02d}-{d:02d}"

PAD, BOS, EOS = 0, 1, 2
itos = ["<pad>", "<bos>", "<eos>"] + sorted(set("abcdefghijklmnopqrstuvwxyz0123456789 ,/.-"))
stoi = {c: i for i, c in enumerate(itos)}
vocab_size = len(itos)
src_len, tgt_len = 32, 12        # max source chars; target = <bos> + 10 chars + <eos>

def make_batch(n):
    # -> src (n, S) padded, tgt (n, 12) = <bos> y y y y - m m - d d <eos>
    pairs = [make_pair() for _ in range(n)]
    src = torch.tensor([[stoi[c] for c in s] + [PAD] * (src_len - len(s)) for s, _ in pairs])
    tgt = torch.tensor([[BOS] + [stoi[c] for c in t] + [EOS] for _, t in pairs])
    return src.to(device), tgt.to(device), pairs

src, tgt, pairs = make_batch(4)
print("vocab:", vocab_size, "|", src.shape, tgt.shape)
for p in pairs:
    print(f"   {p[0]:32s} -> {p[1]}")

## 2. Positional encoding
Attention is a weighted sum: it has no idea of token order. The paper adds a fixed vector to each embedding, built from sines and cosines of different frequencies.

### Formula: Sinusoidal positional encoding
$$PE_{(pos,\, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/D}}\right), \qquad PE_{(pos,\, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/D}}\right)$$
- $pos$: token position; $i = 0, \dots, D/2 - 1$: index of the sin/cos pair; $D = 128$: model dim
- wavelengths form a geometric progression from $2\pi$ to $10000 \cdot 2\pi$
- $PE_{pos + k}$ is a linear function (a rotation) of $PE_{pos}$, so relative offsets are easy to attend to. RoPE (`rope.ipynb`) applies that rotation directly to queries and keys instead
- no parameters; the paper found learned positions work equally well

### Formula: Input embedding
$$h_0 = \text{Dropout}\big(\sqrt{D}\, E[x] + PE\big)$$
- $E \in \mathbb{R}^{V \times D}$: embedding table, **shared** by the encoder input, the decoder input and (transposed) the output layer
- $\sqrt{D}$: scales the embeddings up so the positional signal does not drown them

In [ ]:
class PositionalEncoding(nn.Module):
    # (B, T, D) -> (B, T, D)
    def __init__(self, dim, max_len=512, dropout=0.1):
        super().__init__()
        pos = torch.arange(max_len).float()[:, None]                              # (max_len, 1)
        div = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))  # (D/2,) = 1 / 10000^(2i/D)
        pe = torch.zeros(max_len, dim)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)                                            # (max_len, D), not a parameter
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(x + self.pe[:x.shape[1]])

plt.figure(figsize=(9, 2.5))
plt.imshow(PositionalEncoding(128).pe[:64].T, aspect="auto", cmap="RdBu")
plt.xlabel("position"); plt.ylabel("dim"); plt.show()

## 3. Multi-head attention (self and cross)
One module serves all three uses; they differ only in where $Q, K, V$ come from and which mask is applied.

| use | queries from | keys / values from | mask |
|---|---|---|---|
| encoder self-attention | source | source | source padding |
| decoder self-attention | target | target | **causal** + target padding |
| decoder **cross-attention** | target | encoder output (memory) | source padding |

### Formula: Scaled dot-product attention
$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$
- $Q \in \mathbb{R}^{T_q \times d_k}$, $K, V \in \mathbb{R}^{T_k \times d_k}$: per head; $d_k = D / h = 32$
- $\sqrt{d_k}$: keeps the scores at unit variance so the softmax does not saturate
- $M$: 0 where attention is allowed, $-\infty$ where it is not

### Formula: Multi-head attention
$$\text{MHA}(X_q, X_{kv}) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)\, W_O, \qquad \text{head}_i = \text{Attention}(X_q W_Q^i,\ X_{kv} W_K^i,\ X_{kv} W_V^i)$$
- $h = 4$ heads; self-attention is the case $X_q = X_{kv}$

In [ ]:
class MultiHeadAttention(nn.Module):
    # q (B, Tq, D), kv (B, Tk, D), mask broadcastable to (B, 1, Tq, Tk) with True = blocked -> (B, Tq, D)
    def __init__(self, dim, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads, self.head_dim = num_heads, dim // num_heads
        self.wq, self.wk, self.wv, self.wo = (nn.Linear(dim, dim) for _ in range(4))
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, kv, mask=None):
        B, Tq, D = q.shape
        Tk = kv.shape[1]
        split = lambda x, T: x.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)   # (B, heads, T, hd)
        Q, K, V = split(self.wq(q), Tq), split(self.wk(kv), Tk), split(self.wv(kv), Tk)

        att = Q @ K.transpose(-2, -1) / self.head_dim ** 0.5          # (B, heads, Tq, Tk)
        if mask is not None:
            att = att.masked_fill(mask, float("-inf"))
        att = att.softmax(dim=-1)
        self.last_att = att.detach()                                  # kept for the attention plot
        out = (self.dropout(att) @ V).transpose(1, 2).reshape(B, Tq, D)
        return self.wo(out)

## 4. Encoder and decoder layers
Each sub-layer is wrapped as `LayerNorm(x + Dropout(Sublayer(x)))`: **Post-LN**, the paper's arrangement. (Later models moved the norm *before* the sub-layer, Pre-LN, which trains more stably at depth; see `vit.ipynb`, `slm.ipynb`.)

### Formula: Encoder layer
$$x' = \text{LN}\big(x + \text{MHA}(x, x)\big), \qquad x_{out} = \text{LN}\big(x' + \text{FFN}(x')\big)$$

### Formula: Decoder layer
$$y' = \text{LN}\big(y + \text{MHA}_{causal}(y, y)\big), \qquad y'' = \text{LN}\big(y' + \text{MHA}(y', m)\big), \qquad y_{out} = \text{LN}\big(y'' + \text{FFN}(y'')\big)$$
- $m$: the encoder's final output ("memory"), the same for every decoder layer
- the causal mask stops position $t$ from seeing positions $> t$, so all target positions can be trained in parallel

### Formula: Position-wise feed-forward
$$\text{FFN}(x) = W_2\, \text{ReLU}(W_1 x + b_1) + b_2$$
- $W_1 \in \mathbb{R}^{4D \times D}$, $W_2 \in \mathbb{R}^{D \times 4D}$; applied to every position independently

In [ ]:
class FeedForward(nn.Sequential):
    def __init__(self, dim, dropout=0.1):
        super().__init__(nn.Linear(dim, 4 * dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(4 * dim, dim))


class EncoderLayer(nn.Module):
    # x (B, S, D) -> (B, S, D)
    def __init__(self, dim, num_heads, dropout=0.1):
        super().__init__()
        self.attn, self.ffn = MultiHeadAttention(dim, num_heads, dropout), FeedForward(dim, dropout)
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        x = self.ln1(x + self.dropout(self.attn(x, x, src_mask)))       # self-attention
        return self.ln2(x + self.dropout(self.ffn(x)))


class DecoderLayer(nn.Module):
    # y (B, T, D), memory (B, S, D) -> (B, T, D)
    def __init__(self, dim, num_heads, dropout=0.1):
        super().__init__()
        self.self_attn, self.cross_attn = MultiHeadAttention(dim, num_heads, dropout), MultiHeadAttention(dim, num_heads, dropout)
        self.ffn = FeedForward(dim, dropout)
        self.ln1, self.ln2, self.ln3 = nn.LayerNorm(dim), nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, y, memory, tgt_mask, src_mask):
        y = self.ln1(y + self.dropout(self.self_attn(y, y, tgt_mask)))            # masked self-attention
        y = self.ln2(y + self.dropout(self.cross_attn(y, memory, src_mask)))      # queries: target, keys/values: source
        return self.ln3(y + self.dropout(self.ffn(y)))

## 5. The full model
$N = 2$ encoder and 2 decoder layers, $D = 128$, 4 heads (paper base model: $N = 6$, $D = 512$, 8 heads, 65M params). `encode` runs once per source; `decode` is called with the target prefix.

In [ ]:
class Transformer(nn.Module):
    # src (B, S), tgt (B, T) -> logits (B, T, V)
    def __init__(self, vocab_size, dim=128, depth=2, num_heads=4, dropout=0.1):
        super().__init__()
        self.dim = dim
        self.embed = nn.Embedding(vocab_size, dim)
        self.pos = PositionalEncoding(dim, dropout=dropout)
        self.encoder = nn.ModuleList([EncoderLayer(dim, num_heads, dropout) for _ in range(depth)])
        self.decoder = nn.ModuleList([DecoderLayer(dim, num_heads, dropout) for _ in range(depth)])
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        nn.init.normal_(self.embed.weight, std=dim ** -0.5)       # shared with the output layer

    def encode(self, src):
        src_mask = (src == PAD)[:, None, None, :]                  # (B, 1, 1, S): hide padded source keys
        x = self.pos(self.embed(src) * self.dim ** 0.5)            # (B, S, D)
        for layer in self.encoder:
            x = layer(x, src_mask)
        return x, src_mask

    def decode(self, tgt, memory, src_mask):
        T = tgt.shape[1]
        causal = torch.ones(T, T, dtype=torch.bool, device=tgt.device).triu(1)   # (T, T) True above the diagonal
        tgt_mask = causal[None, None] | (tgt == PAD)[:, None, None, :]           # (B, 1, T, T)
        y = self.pos(self.embed(tgt) * self.dim ** 0.5)            # (B, T, D)
        for layer in self.decoder:
            y = layer(y, memory, tgt_mask, src_mask)
        return y @ self.embed.weight.T                             # (B, T, V): output layer tied to the embedding

    def forward(self, src, tgt):
        memory, src_mask = self.encode(src)
        return self.decode(tgt, memory, src_mask)

model = Transformer(vocab_size).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(model(src, tgt[:, :-1]).shape)

## 6. Training
**Teacher forcing**: the decoder input is the target shifted right (`<bos> 2 0 1 9 - ...`), and it must predict the target shifted left (`2 0 1 9 - ... <eos>`). Every position is trained in one pass.

### Formula: Label smoothing
$$q_i = \begin{cases} 1 - \epsilon_{ls} & i = y \\ \epsilon_{ls} / (V - 1) & i \ne y \end{cases}, \qquad \mathcal{L} = -\sum_i q_i \log p_i$$
- $\epsilon_{ls} = 0.1$: the target distribution keeps 0.9 on the true token and spreads 0.1 over the rest
- stops the model from pushing logits to infinity; the paper notes it hurts perplexity but improves accuracy and BLEU
- (PyTorch's `label_smoothing` spreads $\epsilon / V$ over all classes including the true one; the difference is negligible)

### Formula: Learning-rate schedule ("Noam")
$$\eta_t = D^{-0.5} \cdot \min\big(t^{-0.5},\ t \cdot T_w^{-1.5}\big)$$
- linear warmup for $T_w$ steps (400 here, 4000 in the paper), then decay $\propto 1 / \sqrt{t}$
- peak at $t = T_w$: $\eta = 1 / \sqrt{D \cdot T_w}$
- warmup matters for Post-LN: with a large learning rate from step 0, Post-LN Transformers often fail to train

Adam with the paper's $\beta_1 = 0.9$, $\beta_2 = 0.98$, $\epsilon = 10^{-9}$.

In [ ]:
max_iters = 3000
batch_size = 128
warmup = 400
eval_every = 500

optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)   # lr = 1: the schedule sets the value
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda t: model.dim ** -0.5 * min((t + 1) ** -0.5, (t + 1) * warmup ** -1.5))

@torch.no_grad()
def greedy_decode(model, src):
    # src (B, S) -> generated target (B, 12) including <bos>
    model.eval()
    memory, src_mask = model.encode(src)                                  # encoder runs once
    out = torch.full((src.shape[0], 1), BOS, device=src.device)
    for _ in range(tgt_len - 1):
        logits = model.decode(out, memory, src_mask)                      # (B, t, V)
        out = torch.cat([out, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    model.train()
    return out

@torch.no_grad()
def exact_match(model, n=1000):
    src, tgt, _ = make_batch(n)
    return (greedy_decode(model, src) == tgt).all(dim=1).float().mean().item()   # whole date must be right

losses = []
model.train()
for it in range(1, max_iters + 1):
    src, tgt, _ = make_batch(batch_size)
    logits = model(src, tgt[:, :-1])                                      # decoder input: all but the last token
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt[:, 1:].reshape(-1), ignore_index=PAD, label_smoothing=0.1)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    losses.append(loss.item())
    if it % eval_every == 0:
        print(f"iter {it:5d} | loss {sum(losses[-100:]) / 100:.4f} | lr {scheduler.get_last_lr()[0]:.2e} | exact match {exact_match(model):.3f}")

plt.plot(losses); plt.xlabel("iter"); plt.ylabel("train loss (label-smoothed)"); plt.show()

## 7. Translate
Greedy decoding: run the encoder once, then feed the decoder its own output one token at a time. (The paper uses **beam search** with beam 4, keeping the 4 best partial translations instead of 1.)

In [ ]:
decode_ids = lambda ids: "".join(itos[i] for i in ids if i > EOS)

examples = ["5 march 2019", "december 25, 1999", "saturday, july 4, 2026", "09/11/01", "1.2.2003", "feb 14 1987"]
src = torch.tensor([[stoi[c] for c in s] + [PAD] * (src_len - len(s)) for s in examples], device=device)
out = greedy_decode(model, src)
for s, o in zip(examples, out.tolist()):
    print(f"{s:28s} -> {decode_ids(o)}")

## 8. Cross-attention: which source characters does each output look at?
Rows: output characters as they are generated. Columns: source characters. Last decoder layer, averaged over heads. For the year digits the mass should sit on the year in the source, for the month digits on the month name, and so on: the model has learned the **alignment** without ever being told it.

In [ ]:
s = "saturday, july 4, 2026"
src = torch.tensor([[stoi[c] for c in s]], device=device)                 # (1, S), no padding
out = greedy_decode(model, src)                                           # fills last_att of every attention module
att = model.decoder[-1].cross_attn.last_att[0].mean(0).cpu()              # (heads, T, S) -> (T, S)

plt.figure(figsize=(9, 4))
plt.imshow(att, aspect="auto", cmap="viridis")
plt.xticks(range(len(s)), list(s)); plt.yticks(range(att.shape[0]), [itos[i] for i in out[0, 1:].tolist()])
plt.xlabel("source"); plt.ylabel("generated"); plt.colorbar(); plt.show()

## Notes
- The paper's results: 28.4 BLEU on WMT14 English-German and 41.8 on English-French (`bleu.ipynb`), at a fraction of the training cost of the recurrent and convolutional models it replaced.
- Cost: self-attention is $O(T^2 \cdot D)$ per layer against an RNN's $O(T \cdot D^2)$, but with $O(1)$ sequential steps instead of $O(T)$. That parallelism, not accuracy alone, is why it took over.
- The two halves went separate ways: the **encoder** became BERT (`bert.ipynb`), the **decoder** without cross-attention became GPT (`slm.ipynb`). Encoder-decoder models are still used where there is a clear input -> output mapping: T5, BART, Whisper (speech -> text), and the original use, machine translation.
- What changed since: Pre-LN instead of Post-LN, RMSNorm instead of LayerNorm, GELU / SwiGLU instead of ReLU, RoPE instead of added sinusoids, AdamW with warmup + cosine decay instead of the Noam schedule. The attention formula itself is unchanged.
- Generation here re-runs the decoder on the whole prefix each step. Real implementations cache the keys and values of past tokens (**KV cache**, `llama2.ipynb`); the encoder output is computed once either way.